# Snake venom classification exercise

**Author/contact:** Jonathan Funk ([jonfu@dtu.dk](mailto:jonfu@dtu.dk))


In this exercise, we are going to take a look at snake venoms :snake:. Because this is an educational course, we are first going
to examine how we can use vsiualizations and basic machine learning libraries, such as scikit-learn to classify the snake venoms.
Later we will revisit the same workflow with reusable scikit-learn utilities and compare several model choices.

We will use the UniProt database to fetch snake venom toxin sequences. We will use a custom query to select reviewed snake toxin entries.

**UniProt Query**: `taxonomy_id:8570 AND ( cc_tissue_specificity:venom OR cc_scl_term:SL-0177) AND reviewed:true`

We filtered the data, removing toxins that are hard to group into categories and assigned Function Classes based on the 'Function [CC]' column
using OpenAIs GPT o1 model. This was done to create simple functional labels that we can more easily work with, instead of the lengthy function descriptions.

Pandas is imported to load the data and Matplotlib is used to visualize it. Take a look at the data columns.

_The exercises have been created by Jonathan Funk and Valentas Brasas_


## Learning goals and working style

By the end of this exercise you should be able to explain, without looking at the code, how a protein sequence becomes a feature matrix, why data splitting matters, and how to read a confusion matrix when classes are imbalanced.

Work in pairs when possible. Before you run a model, make a short prediction about what you expect to happen. After you run it, compare the result with your prediction and write down one thing that surprised you.

In [ ]:
import pandas as pd

df = pd.read_csv("Snake_Toxins_with_Function_Classes.csv")
df

### Exercise 1: Exploratory Data Analysis (EDA)

Objective: Familiarizing with the dataset and understanding the distribution of different toxin functional classes to identify any class imbalances that might affect model performance.


1. Count the Number of Instances per Class
2. Check if there are any missing, duplicate or inconsistent data to ensure data quality for model training.
3. Analyze the distribution of protein sequence lengths to understand variability and inform preprocessing steps like padding or truncation.
4. Examine the frequency and distribution of each amino acid in the protein sequences to identify patterns or biases that could inform feature engineering.

### Check your understanding with a neighbor

Before writing code, discuss these questions for two minutes:

1. Which column do you think is the biological input, and which column is the label we want to predict?
2. If one functional class has many more examples than another, how could a model appear accurate while still being scientifically unhelpful?
3. What would count as data leakage in this dataset?

## Visualizing protein sequences

In machine learning data is represented numerically, frequently in the form of vectors. There are countless different choices to be made
when representing proteins as vectors, which will be covered in detail in later lectures. Here we are going to start simply by encoding
sequences as one-hot encoded vectors. This means, that we are going to assign each residue in the sequence with a vector, that is
0 in every position except for a single 1. The position of the one will indicate which amino acid we are dealing with.

For example, the amino acid Alanine (Ala, A) can be represented as having the 1 in the first position, while the amino acid
Cystein (Cys, C) can be represented as having the 1 in the second position. Thus, we will communicate that the two amino acids
are different entities. This method of encoding is known as One-Hot Encoding, or short OHE and commonly used to represent discrete
sequences.

Note, that the only information the machine learning algorithms will have when using this annotation is, that
the amino acids are different.

### Ecercise 2 : What could be a problem when representing amino acids as:

    1. A=1, C=2, D=3, ..., Y=20?
    
    2. Using OHE

We will use numpy to encode the protein sequences as vectors

### Prediction before implementation

Sketch what the one-hot matrix for a short sequence such as `ACDC` should look like before running the code. Then explain to your neighbor what information one-hot encoding preserves and what information it throws away.

In [ ]:
import numpy as np


def one_hot_encode(seq):
    # Dictionary of standard amino acids
    amino_acids = "ACDEFGHIKLMNPQRSTVWY"
    aa_to_int = {aa: i for i, aa in enumerate(amino_acids)}

    # Initialize the one-hot encoded matrix
    one_hot = np.zeros((len(seq), len(amino_acids)), dtype=int)

    # Fill the matrix
    for i, aa in enumerate(seq):
        if aa in aa_to_int:
            one_hot[i, aa_to_int[aa]] = 1

    return one_hot

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


def visualize_one_hot(encoded_seq, seq):
    # Create a heatmap from the one-hot encoded matrix
    plt.figure(figsize=(10, len(encoded_seq) / 2))
    sns.heatmap(
        encoded_seq,
        annot=True,
        cbar=False,
        cmap="rocket",
        xticklabels=list("ACDEFGHIKLMNPQRSTVWY"),
        yticklabels=list(seq),
    )
    plt.xlabel("Amino Acids")
    plt.ylabel("Sequence Position")
    plt.title("One-Hot Encoding of Amino Acid Sequence")
    plt.savefig("one_hot_encoding.png")
    plt.show()


# Example usage
seq = "MLQVLLVTICLAVF"
encoded_seq = one_hot_encode(seq)
visualize_one_hot(encoded_seq, seq)

### Now let's encode the entire df

In machine learning it is common to call the input x, thus we are encoding the sequences and give the column the name x.

### Exercise 3:

Create a column 'x' in the dataframe to encode all protein sequences using one-hot encoding.

### Reflection prompt

After creating `x`, inspect a few entries. Are all encoded sequences the same shape? If not, decide whether you want to pad, truncate, flatten, or switch to a length-independent representation such as k-mers. Explain the tradeoff you chose.

## Encoding class labels

Next we also need to encode the class labels. These are also going to be encoded using OHe, however, the library we are going to use expects train the classification model
expects simple numbers as class labels and does the OHE under the hood. Because of this we are going to encode the classes as numbers. The output variable (class in this case),
is often called y. Thus we are calling the encoded class labels y.

### Exercise 4:

Create a column 'y' in the dataframe, assigning unique integer values to individual function classes:

### Misconception check

The integer labels are identifiers, not measurements. Explain why class `4` is not twice as large as class `2`, and why this matters when choosing a loss function or model.

## Training the classification model using scikit-learn

Next, we are going to train machine learning models using scikit-learn to classify snake toxins based on.

### Exercise 5

    3.1 Load the dataset into variables X (features) and y (labels).
    3.2 Explore the structure of X. Are all sequences in X the same size? If not, figure out how to handle this.
    3.3 Divide the dataset into training and testing sets.
    3.4 Pick a classifier from the following options:
        a. Logistic Regression
        b. Random Forest
        c. Support Vector Machine
    3.5 Make predictions
    3.6 Evaluate the model with appropriate metrics and interpret the results
    

### Before you fit a model

Write down a baseline first: what accuracy or macro-F1 score would you expect from always predicting the most common class? Use that baseline when judging whether your trained model learned anything useful.

## Visualizing results

Now that we have trained the model we would like to visualize the results, which is an important step to communicate the capabilities of your model, but its also interesting for you to quickly see where your model performs better or worse. Visualize your results using a confusion matrix.

### Exercise 6

    4.1 Visualize the classification results using a confusion matrix. 
    4.2 Discuss the results? What are the differences between the classes?

### Interpret before optimizing

Pick one off-diagonal cell in the confusion matrix and discuss a possible biological or data-driven reason for that mistake. Is it caused by similar sequence patterns, class imbalance, noisy labels, or a weak representation?

### Exercise 7

1. Compare Metrics: Write a code to analyze and compare how different classification metrics (precision, recall, F1-score) correlate with each other.

2. Add a New Metric: Implement an additional metric (e.g., ROC-AUC, Matthews Correlation Coefficient) and use it to further compare the models.

3. Model Comparison: Re-run three classification models—Logistic Regression, Random Forest, and Support Vector Machine (SVM). Compare their performance using the metrics above and determine which model performs best.

4. Optimize Model Performance: Explore and apply new strategies to improve classification results. Potential strategies to try:
   * Experiment with different sequence encoding methods (e.g., k-mer encoding, * physicochemical properties).
   * Address class imbalance using techniques like oversampling, undersampling, or adjusting class weights.
   * Implement an ensemble method, such as a voting classifier or boosting, to combine the strengths of multiple models.
   * Perform hyperparameter tuning using GridSearchCV, RandomizedSearchCV, or Bayesian optimization.



# Reusable scikit-learn workflow

Now let's repeat the same classification task with explicit scikit-learn components. The goal is to keep every preprocessing and modeling step visible while avoiding proprietary helper libraries.

### Explain the pipeline

In pairs, describe what each pipeline step does using non-code language. One person should explain `CountVectorizer`; the other should explain `RandomForestClassifier`. Switch roles and correct each other.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.manifold import TSNE
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

X_seq = df["Sequence"].astype(str)
y_label = df["Function Class"].astype(str)

X_train, X_test, y_train, y_test = train_test_split(
    X_seq,
    y_label,
    test_size=0.2,
    random_state=42,
    stratify=y_label,
)

kmer_rf = Pipeline([
    ("kmers", CountVectorizer(analyzer="char", ngram_range=(3, 3))),
    ("classifier", RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced", n_jobs=-1)),
])

kmer_rf.fit(X_train, y_train)
y_pred = kmer_rf.predict(X_test)
print(classification_report(y_test, y_pred))

## Visualize k-mer sequence features

The vectorizer turns each sequence into counts of amino-acid 3-mers. We can visualize these open features directly with t-SNE.

In [ ]:
X_kmer = kmer_rf.named_steps["kmers"].transform(X_seq)
perplexity = min(30, max(5, (len(df) - 1) // 10))
xy = TSNE(n_components=2, perplexity=perplexity, init="random", learning_rate="auto", random_state=42).fit_transform(X_kmer)

plot_df = pd.DataFrame({"tSNE-1": xy[:, 0], "tSNE-2": xy[:, 1], "Function Class": y_label})
plt.figure(figsize=(10, 7))
sns.scatterplot(data=plot_df, x="tSNE-1", y="tSNE-2", hue="Function Class", s=25, alpha=0.8)
plt.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0))
plt.tight_layout()
plt.savefig("tsne.png", dpi=200)
plt.show()

## Confusion matrix

A confusion matrix shows which functional classes are being mixed up by the model.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    xticks_rotation=90,
    cmap="Blues",
    ax=ax,
)
plt.tight_layout()
plt.show()

## Optional extension: compare model families

Use the same k-mer features with multiple classifiers. This mirrors realistic model selection without hiding the preprocessing steps.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, matthews_corrcoef, precision_score, recall_score
from sklearn.svm import LinearSVC

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced", n_jobs=-1),
    "Linear SVM": LinearSVC(class_weight="balanced"),
}

rows = []
for name, clf in models.items():
    pipe = Pipeline([
        ("kmers", CountVectorizer(analyzer="char", ngram_range=(3, 3))),
        ("classifier", clf),
    ])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    rows.append({
        "model": name,
        "precision_macro": precision_score(y_test, pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_test, pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_test, pred, average="macro", zero_division=0),
        "mcc": matthews_corrcoef(y_test, pred),
    })

pd.DataFrame(rows).sort_values("f1_macro", ascending=False)